# Day 11: 遺伝子IDとアノテーション

## この章で解く現実の課題

`ENSG00000141510` と `TP53` はどう対応し、解析表ではどちらを使うべきかを考えます。

RNA-seq解析では、計算に安定したIDを使い、解釈には人間が読みやすいgene symbolを使うことがよくあります。この対応づけを雑に行うと、別の遺伝子として解釈したり、重複で集計を誤ったりします。


## 最小限の用語

- Ensembl gene ID: `ENSG...` のような安定ID。
- version suffix: `ENSG00000141510.18` の `.18` の部分。参照バージョンを表す。
- gene symbol: `TP53` のような人間が読みやすい名前。
- annotation table: ID、symbol、遺伝子説明などを対応づける表。


In [ ]:
import pandas as pd

results = pd.DataFrame({
    "ensembl_id": ["ENSG00000136244.11", "ENSG00000141510.18", "ENSG00000163735.9", "ENSG00000232810.5"],
    "log2_fold_change": [5.1, -1.2, 2.4, 3.8],
    "padj": [0.0001, 0.03, 0.001, 0.02],
})

annotation = pd.DataFrame({
    "ensembl_id_clean": ["ENSG00000136244", "ENSG00000141510", "ENSG00000163735", "ENSG00000232810"],
    "symbol": ["IL6", "TP53", "CXCL8", "TNF"],
    "description": [
        "interleukin 6; inflammatory cytokine",
        "tumor protein p53; stress response regulator",
        "chemokine involved in neutrophil recruitment",
        "tumor necrosis factor; inflammatory cytokine",
    ],
})

results


In [ ]:
# Ensembl IDにはバージョンが付くことがある。
# annotation表とjoinする前に、.以降を外してIDの形をそろえる。
results["ensembl_id_clean"] = results["ensembl_id"].str.replace(r"\..*$", "", regex=True)
annotated = results.merge(annotation, on="ensembl_id_clean", how="left")
annotated


In [ ]:
# 解釈しやすい順に並べる。
# 計算列は残しつつ、symbolとdescriptionを加えると結果表として読める。
report_table = annotated[["symbol", "ensembl_id", "log2_fold_change", "padj", "description"]].sort_values("padj")
report_table


## 読み取り

`ENSG00000141510.18` はバージョン付きのEnsembl IDです。annotation表がバージョンなしIDを持っている場合、そのまま結合すると対応しません。解析ではこのようなID形式の違いが頻繁に起きます。

また、gene symbolは変わることがあります。同じsymbolが複数IDに対応することもあります。計算の主キーには安定IDを残し、表示用にsymbolを加えるのが安全です。


In [ ]:
checks = pd.DataFrame({
    "check": ["unmatched IDs", "duplicated symbols", "version suffix removed"],
    "value": [
        annotated["symbol"].isna().sum(),
        annotation["symbol"].duplicated().sum(),
        results["ensembl_id"].str.contains(r"\.").any(),
    ],
})
checks


## エラーが出た場合

- `KeyError: ensembl_id_clean`: ID整形セルを先に実行してください。
- `symbol` が `NaN`: annotation表に対応IDがないか、IDバージョンの扱いが合っていません。

## この章のゴール

計算用IDと解釈用symbolを区別し、安全に結果表へアノテーションを追加できれば合格です。
